# Budget check on Colab

Runs one development pair of length-2 chain problems through the single agent, the ensemble, and the hierarchy at 4k, 8k, and 16k tokens. The goal is to find the smallest budget where the ensemble attempts and hierarchy stages stop hitting their token caps, and to measure how long each run takes.

Use **Runtime > Change runtime type > L4 GPU**. Start from a fresh runtime (**Runtime > Disconnect and delete runtime**) if an earlier version of this notebook installed `torch==2.8.0`. Results go to `MyDrive/small-multi-agents/results/budget_check/` in Google Drive, with one subfolder per budget holding a log for each system, plus two summary CSV files. If the session disconnects, rerun every cell. Each run continues from its last finished problem.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/small-multi-agents/results/budget_check"
!mkdir -p {OUT}

In [ ]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

This keeps Colab's preinstalled CUDA torch, as `colab.ipynb` does. Installing the pinned `torch==2.8.0` over it breaks Colab's matching torchvision, which transformers imports when loading Qwen3. The versions in use are saved to `environment.txt` next to the results. The first run of each script downloads about 8 GB of Qwen3-4B weights.

In [ ]:
!grep -v '^torch==' requirements.txt > /tmp/colab-requirements.txt
!pip install -q -r /tmp/colab-requirements.txt
!python -c "import torch, transformers; print('torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

## Run

`results/chain_calibration/d2.jsonl` is the calibration pair (generator seed 2027, two updates per chain). Budgets run in increasing order, so the 4k results for all three systems finish first. Each run reloads the model, which adds load time to the cell but not to the per-problem `seconds` in the logs.

In [ ]:
PROBLEMS = "results/chain_calibration/d2.jsonl"
BUDGETS = [4000, 8000, 16000]
SYSTEMS = ["single_agent", "ensemble", "hierarchy"]

for budget in BUDGETS:
    !mkdir -p {OUT}/b{budget}
    for system in SYSTEMS:
        log = f"{OUT}/b{budget}/{system}.jsonl"
        print(f"=== {system}, budget {budget}")
        !python run_{system}.py {PROBLEMS} --budget {budget} --seed 42 --output {log} --resume
        if _exit_code:
            raise RuntimeError(f"{system} at budget {budget} exited with {_exit_code}")

## Results

One row per problem, saved to `summary_problems.csv`. `parts` lists each ensemble attempt's or hierarchy stage's status, which shows whether a split system ran out of tokens. `out_tok_per_s` is generated tokens per second of wall time for the whole problem.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path(OUT).glob("b*/*.jsonl")):
    system = path.stem
    for line in path.read_text().splitlines():
        r = json.loads(line)
        if system == "ensemble":
            parts = [a["status"] for a in r["attempts"]]
        elif system == "hierarchy":
            parts = [f'{c["stage"]}:{c["stop_reason"]}' for c in r["calls"]]
        else:
            parts = []
        rows.append({
            "system": system, "budget": r["budget"], "kind": r["kind"], "status": r["status"],
            "parts": ", ".join(parts), "input": r["input_tokens"], "output": r["output_tokens"],
            "total": r["total_tokens"], "seconds": r["seconds"],
            "out_tok_per_s": round(r["output_tokens"] / r["seconds"], 1) if r["seconds"] else None,
        })

df = pd.DataFrame(rows).sort_values(["budget", "system", "kind"], ascending=[True, True, False])
df.to_csv(f"{OUT}/summary_problems.csv", index=False)
pd.set_option("display.max_colwidth", None)
df

Totals by system and budget, saved to `summary_by_system.csv`.

In [ ]:
by_system = df.groupby(["system", "budget"]).agg(
    correct=("status", lambda s: (s == "correct").sum()),
    problems=("status", "size"),
    mean_total=("total", "mean"),
    total_seconds=("seconds", "sum"),
)
by_system.to_csv(f"{OUT}/summary_by_system.csv")
by_system